# Etapa 5 — Integração do MVP com a camada prata no Google Drive

**Objetivo único desta etapa:** usar o arquivo **já persistido na nuvem** (`ai4i_modelagem.csv`) como entrada dos três classificadores originais; comparar as métricas com as saídas anteriormente reproduzidas e executar a mesma função de apoio à decisão. Não ocorre nova coleta da UCI nesta etapa.

**Fonte:** AI4I 2020, UCI Machine Learning Repository, ID 601, licença CC BY 4.0. **Código de referência:** [repositório do MVP](https://github.com/ricardopimentelandre/MVP-Manutencao-Preditiva), notebook de origem associado ao commit `303a80b0c9a74d098251a78571c1e3aedb2842fc`.

**Pasta e evidências:** leitura de `Meu Drive/UnB/SSD/MVP_Manutencao_Preditiva/execucoes/20261109T200446Z/prata/ai4i_modelagem.csv`, criado na Etapa 2; gravação de **um novo JSON** na pasta `evidencias` da mesma execução.

**Cuidado:** os resultados são demonstrativos. A escolha do modelo usa a mesma partição de teste utilizada para reportar as métricas; não há validação externa ou em máquinas reais. `predict_proba` é um escore do classificador, não uma probabilidade de falha calibrada. Este notebook não altera o CSV existente, o notebook original nem o short paper.

**Como executar:** `Ambiente de execução → Executar tudo`. Caso o Colab solicite reiniciar o ambiente após instalar pacotes, reinicie e execute novamente. Use a **mesma conta pessoal do Drive** da Etapa 2.

In [ ]:
# Instale as mesmas versões utilizadas na reprodução controlada de 09/11/2026.
# Se o Colab solicitar reiniciar o ambiente após a instalação, reinicie e execute novamente.
%pip -q install "scikit-learn==1.8.0" "numpy==2.4.6" "pandas==2.2.3" "scipy==1.16.3" "joblib==1.6.0"

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import sklearn
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print('Python:', platform.python_version())
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
assert sklearn.__version__ == '1.8.0', 'Versão de sklearn diferente de 1.8.0; revise a instalação/reinicie o runtime.'

drive.mount('/content/drive')

# Caminho da execução do ETL que já foi comprovada e documentada.
pasta_execucao = Path('/content/drive/MyDrive/UnB/SSD/MVP_Manutencao_Preditiva/execucoes/20261109T200446Z')
arquivo_prata = pasta_execucao / 'prata' / 'ai4i_modelagem.csv'
assert arquivo_prata.is_file(), (
    f'Não foi encontrado o CSV prata em {arquivo_prata}. '
    'Verifique se o Drive montado corresponde à conta usada na Etapa 2.'
)

# Confirma que esta é exatamente a versão persistida na Etapa 2.
sha_esperado = '72812a8462ee1dcfee89a0c73c41c47524ddef195db6843bcf40247c591dec39'
sha_atual = hashlib.sha256(arquivo_prata.read_bytes()).hexdigest()
assert sha_atual == sha_esperado, (
    f'SHA-256 do CSV diferente do relatório da Etapa 2. Encontrado: {sha_atual}. '
    'Não utilize arquivos alterados sem investigar a divergência.'
)

prata = pd.read_csv(arquivo_prata)
colunas_entradas = ['Type', 'Air temperature', 'Process temperature',
                    'Rotational speed', 'Torque', 'Tool wear']
colunas_esperadas = colunas_entradas + ['Machine failure']
assert list(prata.columns) == colunas_esperadas, f'Colunas inesperadas: {list(prata.columns)}'
assert prata.shape == (10000, 7), f'Dimensões inesperadas: {prata.shape}'
assert prata.isna().sum().sum() == 0, 'O CSV tem valores ausentes.'
assert set(prata['Type'].unique()) == {'L', 'M', 'H'}, 'Categorias inesperadas.'
assert set(prata['Machine failure'].unique()) == {0, 1}, 'Alvo não binário.'
assert int(prata['Machine failure'].sum()) == 339, 'Quantidade de falhas inesperada.'

X = prata[colunas_entradas].copy()
y = prata['Machine failure'].copy()
print('CSV prata relido do Drive:', arquivo_prata)
print('SHA-256:', sha_atual)
print('Registros:', len(X), '| Entradas:', X.shape[1], '| Falhas:', int(y.sum()))

## Treinamento e avaliação — mesmos parâmetros do MVP original

O pré-processamento é ajustado **somente sobre o treino**; os três modelos compartilham a mesma divisão estratificada 80/20, `random_state=42`. A reprodução anterior (Etapa 1) usa os mesmos parâmetros. Aqui a diferença é **a origem dos dados**, que agora é exclusivamente o CSV prata relido do Drive.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
assert X_train.shape[0] == 8000 and X_test.shape[0] == 2000
assert int(y_test.sum()) == 68

categoricas = ['Type']
numericas = ['Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear']

preprocessador_logistica = ColumnTransformer(transformers=[
    ('categorica', OneHotEncoder(handle_unknown='ignore'), categoricas),
    ('numerica', StandardScaler(), numericas)
])
preprocessador_rf = ColumnTransformer(transformers=[
    ('categorica', OneHotEncoder(handle_unknown='ignore'), categoricas)
], remainder='passthrough')

modelo_logistica = Pipeline(steps=[
    ('preprocessamento', preprocessador_logistica),
    ('modelo', LogisticRegression(max_iter=1000, random_state=42))
])
modelo_rf = Pipeline(steps=[
    ('preprocessamento', preprocessador_rf),
    ('modelo', RandomForestClassifier(n_estimators=100, random_state=42))
])
modelo_rf_balanceado = Pipeline(steps=[
    ('preprocessamento', preprocessador_rf),
    ('modelo', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

# Preserva a ordem de treinamento do notebook original.
modelo_logistica.fit(X_train, y_train)
y_pred_logistica = modelo_logistica.predict(X_test)
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_test)
modelo_rf_balanceado.fit(X_train, y_train)
y_pred_rf_balanceado = modelo_rf_balanceado.predict(X_test)

resultados = {}
for nome, yp in [
    ('Regressão Logística', y_pred_logistica),
    ('Random Forest', y_pred_rf),
    ('Random Forest balanceado', y_pred_rf_balanceado),
]:
    resultados[nome] = {
        'acuracia': float(accuracy_score(y_test, yp)),
        'precisao_falha': float(precision_score(y_test, yp, zero_division=0)),
        'recall_falha': float(recall_score(y_test, yp, zero_division=0)),
        'f1_falha': float(f1_score(y_test, yp, zero_division=0)),
        'matriz_confusao': confusion_matrix(y_test, yp, labels=[0,1]).tolist(),
    }

referencia_matrizes = {
    'Regressão Logística': [[1928, 4], [61, 7]],
    'Random Forest': [[1928, 4], [32, 36]],
    'Random Forest balanceado': [[1931, 1], [37, 31]],
}
conferencias = {nome: v['matriz_confusao'] == referencia_matrizes[nome]
               for nome, v in resultados.items()}
todos_conferem = all(conferencias.values())

quadro = pd.DataFrame.from_dict(resultados, orient='index')
print(quadro[['acuracia', 'precisao_falha', 'recall_falha', 'f1_falha']].round(3))
print('\nMatrizes e comparação com o notebook original:')
for nome, valores in resultados.items():
    print(nome, valores['matriz_confusao'], '| confere:', conferencias[nome])
print('\nRESULTADO DOS MODELOS:', 'TODOS CONFEREM' if todos_conferem else 'HÁ DIVERGÊNCIA — NÃO PUBLICAR COMO VALIDAÇÃO')

## Demonstração da função de apoio à decisão

A função conserva **as mesmas duas mensagens** do MVP original. Usamos os registros 1391 e 2997 porque foram os casos de acerto **escolhidos no notebook original**, e não porque representam uma amostra independente. A seleção a partir dos resultados reais do teste não é uma forma de validar a aplicação em produção.

In [ ]:
def avaliar_maquina(tipo, temperatura_ar, temperatura_processo,
                    velocidade_rotacao, torque, desgaste_ferramenta):
    dados_maquina = pd.DataFrame({
        'Type': [tipo],
        'Air temperature': [temperatura_ar],
        'Process temperature': [temperatura_processo],
        'Rotational speed': [velocidade_rotacao],
        'Torque': [torque],
        'Tool wear': [desgaste_ferramenta]
    })
    previsao = int(modelo_rf.predict(dados_maquina)[0])
    score_falha = float(modelo_rf.predict_proba(dados_maquina)[0][1])
    if previsao == 1:
        resultado = 'POSSÍVEL FALHA IDENTIFICADA'
        recomendacao = 'priorizar a avaliação desta condição pela equipe de manutenção.'
    else:
        resultado = 'NÃO FOI IDENTIFICADA FALHA'
        recomendacao = 'manter o acompanhamento das condições operacionais.'
    print('=== SISTEMA DE SUPORTE À DECISÃO ===')
    print(f'Score de falha do modelo: {score_falha:.2%}')
    print('Resultado:', resultado)
    print('Recomendação:', recomendacao)
    return {'classe_prevista': previsao, 'escore_modelo': score_falha,
            'resultado': resultado, 'recomendacao': recomendacao}

print('Exemplo original — caso de falha corretamente identificado:')
caso_com_falha = avaliar_maquina(
    tipo='L', temperatura_ar=298.9, temperatura_processo=310.2,
    velocidade_rotacao=2737, torque=8.8, desgaste_ferramenta=142
)
print('\nExemplo original — caso sem falha corretamente classificado:')
caso_sem_falha = avaliar_maquina(
    tipo='L', temperatura_ar=300.5, temperatura_processo=309.8,
    velocidade_rotacao=1345, torque=62.7, desgaste_ferramenta=153
)
# As saídas são comparadas à demonstração original, e possíveis divergências ficam registradas.
exemplos_conferem = (
    caso_com_falha['classe_prevista'] == 1 and
    abs(caso_com_falha['escore_modelo'] - 0.94) < 1e-9 and
    caso_sem_falha['classe_prevista'] == 0 and
    abs(caso_sem_falha['escore_modelo'] - 0.13) < 1e-9
)
print('\nEXEMPLOS:', 'CONFEREM' if exemplos_conferem else 'DIVERGÊNCIA — VERIFICAR')

## Registro de evidência

Gravaremos um novo arquivo JSON em `evidencias/`, com data, versões, hash do **CSV realmente lido do Drive**, resultados dos três modelos, resultados dos exemplos e veredicto. O arquivo não substitui relatórios anteriores.

**Importante:** este notebook comprova uma execução de treino e inferência usando dados persistidos na nuvem. O treinamento ocorre na memória do Colab, e o projeto **não** implementa serviço de previsão contínuo, otimização de ordens de serviço ou operação industrial.

In [ ]:
pasta_evidencias = pasta_execucao / 'evidencias'
assert pasta_evidencias.is_dir(), f'Pasta de evidências inexistente: {pasta_evidencias}'
instante = datetime.now(timezone.utc)
relatorio = {
    'projeto': 'MVP SSD — Manutenção Preditiva',
    'etapa': 'Integração da camada prata persistida em nuvem com classificação e apoio à decisão',
    'data_execucao_utc': instante.isoformat(),
    'commit_notebook_original': '303a80b0c9a74d098251a78571c1e3aedb2842fc',
    'origem_efetivamente_lida': str(arquivo_prata),
    'fonte_csv': 'Google Drive / camada prata criada na etapa de ETL, sem nova coleta UCI',
    'sha256_prata': sha_atual,
    'ambiente': {
        'python': platform.python_version(), 'scikit-learn': sklearn.__version__,
        'pandas': pd.__version__, 'numpy': np.__version__,
    },
    'dados': {'n_linhas': int(len(prata)), 'n_colunas': int(prata.shape[1]),
              'n_falhas': int(y.sum()), 'n_treino': int(len(X_train)),
              'n_teste': int(len(X_test)), 'n_falhas_teste': int(y_test.sum())},
    'resultados': resultados,
    'comparacao_matrizes_originais': conferencias,
    'todos_modelos_conferem': bool(todos_conferem),
    'exemplos': {'falha_detectada': caso_com_falha, 'sem_falha_detectada': caso_sem_falha},
    'exemplos_conferem': bool(exemplos_conferem),
    'aprovado': bool(todos_conferem and exemplos_conferem),
    'limites': [
        'Dados sintéticos; não demonstra validade em máquinas reais.',
        'Seleção de modelo baseada na própria partição de teste.',
        'Escores predict_proba não calibrados como probabilidades reais.',
        'Demonstração não é um serviço operacional em tempo real.'
    ]
}

# Nome único: não sobrescrever nenhum arquivo anterior do Drive.
marcador = instante.strftime('%Y%m%dT%H%M%S%fZ')
arquivo_relatorio = pasta_evidencias / f'relatorio_integracao_mvp_{marcador}.json'
with arquivo_relatorio.open('x', encoding='utf-8') as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2)

print('\n=== RESULTADO FINAL DA ETAPA ===')
print('INTEGRAÇÃO VERIFICADA' if relatorio['aprovado']
      else 'INTEGRAÇÃO NÃO CONFIRMADA — confira os detalhes no JSON')
print('Relatório salvo no Drive:', arquivo_relatorio)
print('Envie apenas esse JSON para revisão antes de qualquer mudança no GitHub.')